# Sense Check: Clean Panel + Spread & Duration

This notebook inspects the merged dataset produced by `04_02a_merge_spread_duration.py`.
It reports schema, coverage, missingness, and a summary export for review.

In [1]:
from pathlib import Path
import pandas as pd

# Use the notebook folder as the base directory
notebook_dir = Path.cwd()
# Outputs live under the local outputs folder
out_dir = notebook_dir / "outputs"

merged_path = out_dir / "bond_panel_monthly_clean_with_spread_duration.parquet"
if not merged_path.exists():
    raise FileNotFoundError(
        f"Missing {merged_path}. Run 04_02a_merge_spread_duration.py first."
    )

merged = pd.read_parquet(merged_path, engine="fastparquet")
merged.head()

,RIC,Trade Date,price,month_end,yield,price_return,treasury_ret,credit_excess_return,OAS,Duration
0,001055AY8=,2018-11-30,99.653,2018-11-30,4.772,0.021569,-0.000227,0.021795,159.3437,15.857811
1,001055AY8=,2018-12-31,102.415,2018-12-31,4.600,0.027716,0.000258,0.027458,173.1661,15.981275
2,001055AY8=,2019-01-31,106.537,2019-01-31,4.354,0.040248,0.000276,0.039972,153.3297,16.332479
3,001055AY8=,2019-02-28,106.299,2019-02-28,4.367,-0.002234,-0.000668,-0.001566,146.7975,16.235842
4,001055AY8=,2019-03-29,110.436,2019-03-31,4.132,0.038919,0.000428,0.038490,153.0096,16.429810


In [2]:
# Inspect column dtypes and non-null counts
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99103 entries, 0 to 99102
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   RIC                   99103 non-null  object        
 1   Trade Date            99103 non-null  datetime64[ns]
 2   price                 98990 non-null  float64       
 3   month_end             99103 non-null  datetime64[ns]
 4   yield                 98685 non-null  float64       
 5   price_return          99103 non-null  float64       
 6   treasury_ret          99103 non-null  float64       
 7   credit_excess_return  99103 non-null  float64       
 8   OAS                   86294 non-null  float64       
 9   Duration              86294 non-null  float64       
dtypes: datetime64[ns](2), float64(7), object(1)
memory usage: 7.6+ MB


In [3]:
# Coverage: date range, unique bonds, and total rows
merged["month_end"].min(), merged["month_end"].max(), merged["RIC"].nunique(), len(merged)

(Timestamp('2004-07-31 00:00:00'),
 Timestamp('2025-09-30 00:00:00'),
 1720,
 99103)

In [4]:
# Missingness rates (top 10)
merged.isna().mean().sort_values(ascending=False).head(10)

Duration                0.129249
OAS                     0.129249
yield                   0.004218
price                   0.001140
Trade Date              0.000000
RIC                     0.000000
price_return            0.000000
month_end               0.000000
credit_excess_return    0.000000
treasury_ret            0.000000
dtype: float64

In [6]:
# RIC-level missingness for OAS/Duration
missing_ric = (
    merged.assign(missing_oas=merged["OAS"].isna(), missing_dur=merged["Duration"].isna())
    .groupby("RIC")[["missing_oas", "missing_dur"]]
    .mean()
    .sort_values(["missing_oas", "missing_dur"], ascending=False)
)

missing_ric.head(20)

,missing_oas,missing_dur
RIC,,
00287YAY5=,1.0,1.0
031162DQ0=,1.0,1.0
031162DR8=,1.0,1.0
038222AM7=,1.0,1.0
038222AN5=,1.0,1.0
038222AP0=,1.0,1.0
038222AS4=,1.0,1.0
20030NCU3=,1.0,1.0
20030NCY5=,1.0,1.0


In [7]:
# Count of bonds with any missing OAS/Duration
num_any_missing = (missing_ric.max(axis=1) > 0).sum()
num_all_missing = (missing_ric.max(axis=1) == 1).sum()
num_any_missing, num_all_missing

(np.int64(362), np.int64(21))

In [8]:
# Example: RICs with >=50% missing OAS/Duration
high_missing_ric = missing_ric[missing_ric.max(axis=1) >= 0.5]
high_missing_ric.head(20)

,missing_oas,missing_dur
RIC,,
00287YAY5=,1.0,1.0
031162DQ0=,1.0,1.0
031162DR8=,1.0,1.0
038222AM7=,1.0,1.0
038222AN5=,1.0,1.0
038222AP0=,1.0,1.0
038222AS4=,1.0,1.0
20030NCU3=,1.0,1.0
20030NCY5=,1.0,1.0


## Summary (Merged Panel)

- **Rows:** ~99k monthly observations
- **RICs:** ~1,720
- **Date range:** 2004-07 to 2025-09 (month-end)
- **Missingness:** OAS/Duration ~12.9% (mostly pre-2018), everything else near zero
- **RIC-level missingness:** 362 RICs have some missing OAS/Duration; 21 RICs are fully missing

Cleaning plan:
- **Drop the 21 RICs** with 100% missing OAS/Duration.
- **For the 362 partially missing RICs**, fill OAS/Duration within each RIC over time (ffill/bfill), then drop any remaining gaps.